# Tutorial 1: encoding3d on its own

encoding3d stores a plant model's results as files any platform can read: one file per time step, called a
**frame**. A frame holds the plant's parts at several **scales** (the whole plant, its axes, the small segments
of each axis), how they connect, and the **variables** a model computed on them, each with its unit.

In this tutorial you write such files yourself, with no plant modelling platform at all: a small root system
that grows for five days, made with a few lines of numpy. You will

1. describe the plant at three scales and write one frame;
2. write a frame per day, a **sequence**, and check it;
3. read it back, all at once or only the columns you need;
4. say how ParaView should colour it, and open it there.

Everything is written in `outputs/` next to this notebook.

In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd

from openalea import encoding3d
from openalea.encoding3d import views
from openalea.encoding3d.validate import validate_bytes, validate_sequence

OUTPUTS = "outputs"
os.makedirs(OUTPUTS, exist_ok=True)

## 1. A plant at three scales

Our root system has one **plant**, a few **axes** (the main root and its laterals), and each axis is a chain of
**segments**, 1 cm long, the last one being the growing tip. A toy growth rule (not a real model): every day the main
root grows three segments, each lateral one, and a new lateral starts every second segment of the main root.

Each part gets an **id**, a number that never changes and is never reused: that is how a segment is followed from
one day to the next.

In [ ]:
SEGMENT = 0.01                      # m: every segment is 1 cm long


class ToyRoots:
    """A root system growing by a fixed rule, its parts with stable ids."""

    def __init__(self):
        self.next_id = 1
        self.plant = self.new_id()
        self.axes = {}              # axis id: (order, parent segment id or None, its segment ids)
        self.segments = {}          # segment id: dict(axis, parent, start, end, age)
        self.add_axis(order=1, parent=None, start=np.zeros(3), direction=np.array([0., 0., -1.]))

    def new_id(self):
        self.next_id += 1
        return self.next_id - 1

    def add_axis(self, order, parent, start, direction):
        axis = self.new_id()
        self.axes[axis] = (order, parent, [], direction)
        return axis

    def grow_axis(self, axis, count):
        order, parent, chain, direction = self.axes[axis]
        for _ in range(count):
            previous = chain[-1] if chain else parent
            start = self.segments[previous]["end"] if previous is not None else np.zeros(3)
            segment = self.new_id()
            self.segments[segment] = dict(axis=axis, parent=previous, start=start, end=start + SEGMENT * direction,
                                          age=0.)
            chain.append(segment)

    def day(self):
        for segment in self.segments.values():
            segment["age"] += 1.
        main = next(iter(self.axes))
        self.grow_axis(main, 3)
        for axis, (order, parent, chain, direction) in list(self.axes.items()):
            if order == 2:
                self.grow_axis(axis, 1)
        main_chain = self.axes[main][2]
        bearers = {parent for order, parent, _, _ in self.axes.values() if order == 2}
        for position, segment in enumerate(main_chain[:-1]):
            if position % 2 == 1 and segment not in bearers:
                angle = 2.4 * len(bearers)
                side = np.array([np.cos(angle), np.sin(angle), -0.3])
                lateral = self.add_axis(order=2, parent=segment, start=self.segments[segment]["end"],
                                        direction=side / np.linalg.norm(side))
                self.grow_axis(lateral, 1)
                bearers.add(segment)


roots = ToyRoots()
roots.day()
len(roots.axes), len(roots.segments)

Now the plant as encoding3d tables. Each scale is an **entity set**, a table with one row per part:

- `id`: the part's id;
- `label`: what it is (`RootAxis`, `RootSegment`: names from encoding3d's shared vocabulary);
- `complex`: the part one scale up that contains it (a segment's axis, an axis's plant);
- `parent` and `edge_type`: the part before it at the same scale (a segment's predecessor, or for the first segment
  of a lateral, the segment it branches from: `branch`);
- then the variables, each declared with its **unit** and its **standard name**, so that any platform knows what
  it is (`length` in m, `water_potential` in MPa, converted when needed).

The segments' ends (`x1` … `z2`, declared as `proximal_end_x` … `distal_end_z`) let viewers draw them as tubes.

In [ ]:
def frame_of(roots, day):
    """The root system on *day* as an encoding3d frame."""
    plant_set = encoding3d.EntitySet("Plant", rank=1, columns=dict(
        id=np.array([roots.plant]), label=np.array(["Wheat"], dtype=object)))

    axis_ids = list(roots.axes)
    first_of = {axis: chain[0] for axis, (_, _, chain, _) in roots.axes.items()}
    axis_set = encoding3d.EntitySet("Axis", rank=2, columns=dict(
        id=np.array(axis_ids), label=np.array(["RootAxis"] * len(axis_ids), dtype=object),
        complex=np.full(len(axis_ids), roots.plant),
        parent=np.array([roots.segments[roots.axes[a][1]]["axis"] if roots.axes[a][1] else encoding3d.NO_ID
                         for a in axis_ids]),
        edge_type=np.array(["branch" if roots.axes[a][1] else "none" for a in axis_ids], dtype=object),
        branching_order=np.array([roots.axes[a][0] for a in axis_ids])))
    axis_set.variables["branching_order"] = encoding3d.Variable(unit="1", standard_name="branching_order",
                                                                kind="descriptor")

    ids = list(roots.segments)
    segments = [roots.segments[i] for i in ids]
    starts = np.array([s["start"] for s in segments])
    ends = np.array([s["end"] for s in segments])
    depth = -ends[:, 2]
    tips = {chain[-1] for _, _, chain, _ in roots.axes.values()}
    segment_set = encoding3d.EntitySet("SubOrgan", rank=6, columns=dict(
        id=np.array(ids), label=np.array(["RootApex" if i in tips else "RootSegment" for i in ids], dtype=object),
        complex=np.array([s["axis"] for s in segments]),
        parent=np.array([s["parent"] if s["parent"] is not None else encoding3d.NO_ID for s in segments]),
        edge_type=np.array(["none" if s["parent"] is None else
                            "successor" if roots.segments[s["parent"]]["axis"] == s["axis"] else "branch"
                            for s in segments], dtype=object),
        x1=starts[:, 0], y1=starts[:, 1], z1=starts[:, 2], x2=ends[:, 0], y2=ends[:, 1], z2=ends[:, 2],
        length=np.full(len(ids), SEGMENT),
        radius=0.0005 * (1 + 0.2 * np.array([s["age"] for s in segments])),    # m: thickening with age
        age=np.array([s["age"] for s in segments]),
        water_potential=-0.1 - 2. * depth))                         # MPa: a toy gradient, drier with depth
    for name, standard_name in zip(("x1", "y1", "z1", "x2", "y2", "z2"),
                                   ("proximal_end_x", "proximal_end_y", "proximal_end_z",
                                    "distal_end_x", "distal_end_y", "distal_end_z")):
        segment_set.variables[name] = encoding3d.Variable(unit="m", standard_name=standard_name, kind="descriptor")
    segment_set.variables["length"] = encoding3d.Variable(unit="m", standard_name="length", kind="extensive")
    segment_set.variables["radius"] = encoding3d.Variable(unit="m", standard_name="radius", kind="intensive")
    segment_set.variables["age"] = encoding3d.Variable(unit="d", standard_name="age", kind="intensive")
    segment_set.variables["water_potential"] = encoding3d.Variable(
        unit="MPa", standard_name="water_potential", kind="intensive", description="a toy gradient, lower with depth")

    plant = encoding3d.DataStructure("plant", entity_sets=[plant_set, axis_set, segment_set], platform="tutorial")
    return encoding3d.Frame([plant], time=float(day), time_unit="d", step=day,
                            writer={"platform": "tutorial 1"})


frame = frame_of(roots, 1)
pd.DataFrame(frame.data_structure("plant").entity_set("SubOrgan").columns).head()

A frame becomes a file with `to_bytes` (or `write`). The **validator** checks it against the specification: ids
unique, every `complex` and `parent` found, units valid and convertible to their standard name's.

In [ ]:
data = encoding3d.to_bytes(frame)
print(len(data), "bytes")
print(validate_bytes(data))

A mistake is caught: a length declared in seconds cannot be converted to the standard name's metres.

In [ ]:
wrong = frame_of(roots, 1)
wrong.data_structure("plant").entity_set("SubOrgan").variables["length"].unit = "s"
print(validate_bytes(encoding3d.to_bytes(wrong)))

## 2. A sequence: one frame per day

`write_sequence` writes each frame as a `.glb` file and an index, `scene.fspm.json`, listing their times.

In [ ]:
roots = ToyRoots()
frames = []
for day in range(1, 6):
    roots.day()
    frames.append(frame_of(roots, day))

index = encoding3d.write_sequence(os.path.join(OUTPUTS, "frames"), frames)
print(index)
print(validate_sequence(index))

The command line does the same from a terminal: `encoding3d validate outputs/frames/scene.fspm.json`, or
`encoding3d info` to see what a frame holds.

In [ ]:
info = subprocess.run([sys.executable, "-m", "openalea.encoding3d.cli", "info",
                       os.path.join(OUTPUTS, "frames", "frame_000004.glb")], capture_output=True, text=True)
print(info.stdout)

## 3. Reading it back

`read_sequence` gives the frames one after another. Because ids are stable, you can follow a segment through the
days: here the main root's first segment ages and thickens.

In [ ]:
first_segment = frames[0].data_structure("plant").entity_set("SubOrgan").columns["id"][0]
rows = []
for again in encoding3d.read_sequence(index):
    segments = again.data_structure("plant").entity_set("SubOrgan")
    row = int(np.flatnonzero(segments.columns["id"] == first_segment)[0])
    rows.append(dict(day=again.time, segments=segments.count, first_segment_age=segments.columns["age"][row],
                     first_segment_radius_mm=1000 * segments.columns["radius"][row]))
pd.DataFrame(rows)

For large frames, read **lazily**: only the file's description is parsed at first, and each column is decoded the
first time you use it. The file stays open until the frame is closed (here, at the end of the `with` block).

In [ ]:
with encoding3d.read(os.path.join(OUTPUTS, "frames", "frame_000004.glb"), lazy=True) as last:
    segments = last.data_structure("plant").entity_set("SubOrgan")
    print(segments.count, "segments; columns:", list(segments.columns))
    print("decoded so far:", segments.columns.decoded())
    print("lowest water potential:", segments.columns["water_potential"].min(), segments.variables["water_potential"].unit)
    print("decoded now:", segments.columns.decoded())

## 4. How to look at it: views

A `views.json` next to the frames tells viewers how to draw them: here one **layer**, the segments drawn as tubes
as thick as their radius, coloured by water potential. Viewers that do not know it simply ignore it.

In [ ]:
roots_view = {"encoding3d_views": 1, "active": "default", "configurations": {"default": {"layers": [
    {"name": "roots", "source": {"data_structure": "plant", "kind": "segments", "entity_set": "SubOrgan"},
     "color_by": {"entity_set": "SubOrgan", "variable": "water_potential"}, "colormap": "Viridis (matplotlib)",
     "tube": {"radius": "radius", "scale": 1}}]}}}
views.write_views(os.path.join(OUTPUTS, "frames", views.VIEWS_FILE), roots_view)
print(views.check_views(roots_view, frames[-1]))

**In ParaView** (5.11 or later):

1. once: *Tools → Manage Plugins → Load New*, choose `paraview/plugin_encoding3d.py` of this repository, tick
   *Auto Load*;
2. *File → Open* `tutorials/01_encoding3d_alone/outputs/frames/scene.fspm.json`, choose the *encoding3d reader*,
   *Apply*;
3. press *Play*: the roots grow day by day, coloured as `views.json` says.

`paraview/README.md` explains the rest (switching views, selecting a plant, an axis or a segment).

## Try it yourself

- Make the laterals grow two segments a day, and see the sequence change in ParaView.
- Add a variable of your own to the segments (say, a nitrogen concentration in `mol/m3`), declare it, validate.
- Colour the roots by `age` instead of water potential in `views.json`.
- Tutorial 2 shows the same files written by real platforms: metafspm, VirtualPlantLab, CPlantBox and GroIMP.